<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Processing/colab_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER + DATE/TIME for the GeoRAG corpus
Upload `DATA.json` (or a JSON/JSONL file containing text records). The notebook runs the known BERT baseline for PER/ORG/LOC/MISC and spaCy for DATE/TIME. DATE/TIME are text mentions, not validated event dates.

Start with `LIMIT = 2`, inspect the outputs, then set `LIMIT = None` for the full corpus.


In [1]:
%pip -q install transformers spacy torch
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 52.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import json
import subprocess
from pathlib import Path
import transformers
import spacy
import torch

הגדרת תיקיות

In [3]:
repo_dir = Path("/content/Corpus2GeoRAG_Pacific")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main",
         "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git",
         str(repo_dir)],
        check=True,
    )

In [4]:
data_dir = repo_dir / "Data"
json_files = list(data_dir.glob("*.json"))
if len(json_files) != 1:
    raise ValueError(f"ציפיתי לקובץ JSON אחד בתיקיית Data; נמצאו: {[p.name for p in json_files]}")

קריאה לDATA

In [5]:
input_path = json_files[0]
output_dir = data_dir / "TMP_NER"

with input_path.open(encoding="utf-8") as f:
    data = json.load(f)

print(f"נטען: {input_path.name} — {len(data)} מסמכים")
print(f"פלט יישמר ב: {output_dir}")

נטען: DATA.json — 428 מסמכים
פלט יישמר ב: /content/Corpus2GeoRAG_Pacific/Data/TMP_NER


הצגת 3 הKEYS הראשונים

In [6]:
print(list(data.keys())[:3]) # שלוש הכותרות הראשונות

['14th Indian Infantry Division', '1939–1940 Winter Offensive', '1943 Cairo Declaration']


הגרלת 100 פסקאות

In [7]:
import json
import random
import re

SEED = 42
SAMPLE_SIZE = 100

paragraphs = []

for title, text in data.items():
    separators = list(re.finditer(r"\n[ \t]*\n+", text))
    starts = [0] + [m.end() for m in separators]
    ends = [m.start() for m in separators] + [len(text)]

    for index, (left, right) in enumerate(zip(starts, ends)):
        chunk = text[left:right]
        paragraph = chunk.strip()

        if not paragraph or re.fullmatch(r"=+\s*.*?\s*=+", paragraph):
            continue

        start = left + len(chunk) - len(chunk.lstrip())
        end = left + len(chunk.rstrip())

        paragraphs.append({
            "article_title": title,
            "paragraph_index": index,
            "start": start,
            "end": end,
            "text": text[start:end],
        })

paragraphs.sort(key=lambda p: (p["article_title"], p["start"]))

if SAMPLE_SIZE > len(paragraphs):
    raise ValueError(f"ביקשת {SAMPLE_SIZE} פסקאות, אך נמצאו רק {len(paragraphs)}")

sample = random.Random(SEED).sample(paragraphs, SAMPLE_SIZE)

output_dir.mkdir(parents=True, exist_ok=True)
sample_path = output_dir / f"sample_{SAMPLE_SIZE}_paragraphs_seed{SEED}.json"

result = {
    "source_file": input_path.name,
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "paragraphs": sample,
}

with sample_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"נשמרו {len(sample)} פסקאות מתוך {len(paragraphs)}")
print(sample_path)

נשמרו 100 פסקאות מתוך 9496
/content/Corpus2GeoRAG_Pacific/Data/TMP_NER/sample_100_paragraphs_seed42.json


העלאה לGIT

In [8]:
# import base64
# import requests
# from google.colab import userdata

# token = userdata.get("COLAB")
# if not token:
#     raise ValueError("לא נמצא GITHUB_TOKEN ב-Colab Secrets")

# repo_path = f"Data/TMP_NER/{sample_path.name}"
# url = f"https://api.github.com/repos/danielbr9382/Corpus2GeoRAG_Pacific/contents/{repo_path}"
# headers = {
#     "Authorization": f"Bearer {token}",
#     "Accept": "application/vnd.github+json",
# }
# content = sample_path.read_bytes()

# existing = requests.get(url, headers=headers, params={"ref": "main"}, timeout=30)
# if existing.status_code == 200:
#     remote_content = base64.b64decode(existing.json()["content"])
#     if remote_content != content:
#         raise FileExistsError("ב-GitHub כבר קיים קובץ בשם הזה עם תוכן שונה")
#     print("הקובץ הזה כבר נמצא ב-GitHub")
# elif existing.status_code == 404:
#     response = requests.put(
#         url,
#         headers=headers,
#         json={
#             "message": f"Add NER sample {sample_path.name}",
#             "content": base64.b64encode(content).decode("ascii"),
#             "branch": "main",
#         },
#         timeout=60,
#     )
#     response.raise_for_status()
#     print("הקובץ הועלה ל-GitHub")
# else:
#     existing.raise_for_status()

# print(f"https://github.com/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/{repo_path}")

In [9]:
# error = response.json()

# print("status:", response.status_code)
# print("message:", error.get("message"))
# print("required permissions:",
#       response.headers.get("X-Accepted-GitHub-Permissions"))
# print("rate limit remaining:",
#       response.headers.get("X-RateLimit-Remaining"))

In [8]:
from pathlib import Path
from shutil import copy2

sample_path = Path(sample_path)
output_dir = Path("/content/TMP_NER")
output_dir.mkdir(parents=True, exist_ok=True)

saved_path = output_dir / sample_path.name
if sample_path.resolve() != saved_path.resolve():
    copy2(sample_path, saved_path)

print(f"המדגם נשמר ב־Colab: {saved_path}")

המדגם נשמר ב־Colab: /content/TMP_NER/sample_100_paragraphs_seed42.json


בחינת מודלי NER

In [9]:
def load_sample(path):
    with path.open(encoding="utf-8") as file:
        return json.load(file)

sample_data = load_sample(sample_path)
sample = sample_data["paragraphs"]

print("מספר פסקאות:", len(sample))
print("קובץ:", sample_path)

מספר פסקאות: 100
קובץ: /content/Corpus2GeoRAG_Pacific/Data/TMP_NER/sample_100_paragraphs_seed42.json


In [10]:
def validate_sample(sample, source_data):
    if len(sample) != 100:
        raise ValueError(f"ציפינו ל־100 פסקאות, נמצאו {len(sample)}")

    for paragraph in sample:
        title = paragraph["article_title"]
        start = paragraph["start"]
        end = paragraph["end"]
        text = paragraph["text"]

        if source_data[title][start:end] != text:
            raise ValueError(f"היסטים לא תואמים: {title}, {start}:{end}")

    return True

validate_sample(sample, data)
print("כל 100 הפסקאות תואמות לטקסט המקורי")

כל 100 הפסקאות תואמות לטקסט המקורי


In [11]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

MODEL_ID = "dslim/bert-base-NER"
REVISION = "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, use_fast=True
)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID, revision=REVISION
)

ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="average",
    stride=64,
    device=0 if torch.cuda.is_available() else -1,
)

print("המודל נטען:", MODEL_ID)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


המודל נטען: dslim/bert-base-NER


In [12]:
import pandas as pd
# def extract_mentions(paragraph):
#     text = paragraph["text"]
#     predictions = ner(text)
#     mentions = []

#     for prediction in predictions:
#         start = int(prediction["start"])
#         end = int(prediction["end"])

#         mentions.append({
#             "text": text[start:end],
#             "label": prediction["entity_group"],
#             "start": start,
#             "end": end,
#             "start_document": paragraph["start"] + start,
#             "end_document": paragraph["start"] + end,
#             "score": float(prediction["score"]),
#         })

#     return mentions, predictions

# mentions_first, raw_first = extract_mentions(sample[0])
# pd.DataFrame(mentions_first)

def extract_mentions(paragraph, ner_pipeline=ner):
    text = paragraph["text"]
    predictions = ner_pipeline(text)
    mentions = []

    for prediction in predictions:
        start = int(prediction["start"])
        end = int(prediction["end"])

        mentions.append({
            "text": text[start:end],
            "label": prediction["entity_group"],
            "start": start,
            "end": end,
            "start_document": paragraph["start"] + start,
            "end_document": paragraph["start"] + end,
            "score": float(prediction["score"]),
        })

    return mentions, predictions
mentions_first, raw_first = extract_mentions(sample[0])
pd.DataFrame(mentions_first)

,text,label,start,end,start_document,end_document,score
0,HMS Prince of Wales,ORG,38,57,9052,9071,0.502382
1,HMS Repulse,ORG,77,88,9091,9102,0.787330
2,Singapore,LOC,130,139,9144,9153,0.999773
3,Japanese,MISC,173,181,9187,9195,0.999705
4,Malaya,LOC,210,216,9224,9230,0.999738
5,Japanese,MISC,218,226,9232,9240,0.999593
6,Malayan Peninsula,LOC,330,347,9344,9361,0.684488
7,Japanese,MISC,373,381,9387,9395,0.999658
8,Japanese,MISC,421,429,9435,9443,0.999623
9,Indian,MISC,494,500,9508,9514,0.999735


הרצה על כלל המדגם

In [13]:
bert_results = []

for paragraph in sample:
    mentions, raw = extract_mentions(paragraph, ner)

    bert_results.append({
        "paragraph": paragraph,
        "mentions": mentions,
        "pipeline_output": raw,
    })

print("עובדו", len(bert_results), "פסקאות")

עובדו 100 פסקאות


In [14]:
import json
import uuid
from datetime import datetime, timezone

output_dir = saved_path.parent  # אותה תיקייה של המדגם ב-Colab
model_name = MODEL_ID.replace("/", "_")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid.uuid4().hex[:8]

ner_path = output_dir / f"ner_{model_name}_sample100_seed{sample_data['seed']}_{run_id}.json"

def json_number(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot save {type(value).__name__}")

output = {
    "model_id": MODEL_ID,
    "revision_model": REVISION,
    "sample_file": saved_path.name,
    "seed": sample_data["seed"],
    "results": bert_results,
}

with ner_path.open("x", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2, default=json_number)

print(f"תוצאות ה־NER נשמרו ב־Colab: {ner_path}")

תוצאות ה־NER נשמרו ב־Colab: /content/TMP_NER/ner_dslim_bert-base-NER_sample100_seed42_20261010T094323Z_e601419e.json


מודל Jean-Baptiste/roberta-large-ner-english

טעינת המודל

In [15]:
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

ROBERTA_MODEL_ID = "Jean-Baptiste/roberta-large-ner-english"
ROBERTA_REVISION = model_info(ROBERTA_MODEL_ID).sha

roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_MODEL_ID, revision=ROBERTA_REVISION, use_fast=True
)
roberta_model = AutoModelForTokenClassification.from_pretrained(
    ROBERTA_MODEL_ID, revision=ROBERTA_REVISION
)

roberta_ner = pipeline(
    "ner",
    model=roberta_model,
    tokenizer=roberta_tokenizer,
    aggregation_strategy="average",
    stride=64,
    device=0 if torch.cuda.is_available() else -1,
)

print("המודל נטען:", ROBERTA_MODEL_ID)

config.json:   0%|          | 0.00/849 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

המודל נטען: Jean-Baptiste/roberta-large-ner-english


בדיקה על פסקה אחת

In [16]:
import pandas as pd

roberta_mentions_first, roberta_raw_first = extract_mentions(
    sample[0], roberta_ner
)

pd.DataFrame(roberta_mentions_first)

/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:444: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


,text,label,start,end,start_document,end_document,score
0,Prince of,MISC,42,51,9056,9065,0.745831
1,HMS Repulse,MISC,77,88,9091,9102,0.670890
2,Singapore,LOC,130,139,9144,9153,0.999519
3,Japanese,MISC,173,181,9187,9195,0.999940
4,Malaya.,LOC,210,217,9224,9231,0.666365
5,Japanese,MISC,218,226,9232,9240,0.999889
6,Malayan Peninsula,LOC,330,347,9344,9361,0.993527
7,Japanese,MISC,373,381,9387,9395,0.999891
8,Japanese,MISC,421,429,9435,9443,0.999896
9,Indian,MISC,494,500,9508,9514,0.999920


הרצה על כל הפסקאות

In [17]:
roberta_results = []

for number, paragraph in enumerate(sample, start=1):
    mentions, raw = extract_mentions(paragraph, roberta_ner)

    roberta_results.append({
        "paragraph": paragraph,
        "mentions": mentions,
        "pipeline_output": raw,
    })

    if number % 10 == 0:
        print(f"עובדו {number}/{len(sample)} פסקאות")

print("מספר פסקאות בתוצאות:", len(roberta_results))

/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:444: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


עובדו 10/100 פסקאות
עובדו 20/100 פסקאות
עובדו 30/100 פסקאות
עובדו 40/100 פסקאות
עובדו 50/100 פסקאות
עובדו 60/100 פסקאות
עובדו 70/100 פסקאות
עובדו 80/100 פסקאות
עובדו 90/100 פסקאות
עובדו 100/100 פסקאות
מספר פסקאות בתוצאות: 100


שמירת המדגם

In [18]:
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

output_dir = Path("/content/TMP_NER")
local_sample_path = output_dir / Path(sample_path).name

if not local_sample_path.is_file():
    raise FileNotFoundError(f"קובץ המדגם לא נמצא: {local_sample_path}")

if len(roberta_results) != sample_data["sample_size"]:
    raise ValueError("ההרצה לא הסתיימה על כל המדגם")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid.uuid4().hex[:8]

model_name = ROBERTA_MODEL_ID.replace("/", "_")
result_path = output_dir / (
    f"ner_{model_name}_{local_sample_path.stem}_{run_id}.json"
)

def json_number(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot save {type(value).__name__}")

output = {
    "model_id": ROBERTA_MODEL_ID,
    "revision_model": ROBERTA_REVISION,
    "sample_file": local_sample_path.name,
    "seed": sample_data["seed"],
    "aggregation_strategy": "average",
    "stride_tokens": 64,
    "results": roberta_results,
}

with result_path.open("x", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2,
              default=json_number)

print("התוצאות נשמרו:", result_path)

התוצאות נשמרו: /content/TMP_NER/ner_Jean-Baptiste_roberta-large-ner-english_sample_100_paragraphs_seed42_20261010T095016Z_c6cb4c40.json


המודל הבא הוא spaCy en_core_web_trf

התקנת המודל

In [19]:
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 810.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 86.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


טעינת המודל

In [1]:
import spacy

SPACY_MODEL_ID = "en_core_web_trf"

spacy_nlp = spacy.load(
    SPACY_MODEL_ID,
    disable=["tagger", "parser", "attribute_ruler", "lemmatizer"],
)

print("המודל נטען:", SPACY_MODEL_ID)
print("גרסת המודל:", spacy_nlp.meta["version"])

המודל נטען: en_core_web_trf
גרסת המודל: 3.8.0


חילוץ אזכורים מפסקה אחת

In [2]:
import json
from pathlib import Path

sample_path = Path("/content/TMP_NER/sample_100_paragraphs_seed42.json")

with sample_path.open(encoding="utf-8") as file:
    sample_data = json.load(file)

sample = sample_data["paragraphs"]
print("נטענו", len(sample), "פסקאות")

נטענו 100 פסקאות


In [3]:
def extract_spacy_mentions(paragraph):
    text = paragraph["text"]
    doc = spacy_nlp(text)
    mentions = []

    for entity in doc.ents:
        mentions.append({
            "text": entity.text,
            "label": entity.label_,
            "start": entity.start_char,
            "end": entity.end_char,
            "start_document": paragraph["start"] + entity.start_char,
            "end_document": paragraph["start"] + entity.end_char,
            "score": None,
        })

    return mentions

הצגת האזכורים

In [4]:
import pandas as pd

spacy_mentions_first = extract_spacy_mentions(sample[0])
pd.DataFrame(spacy_mentions_first)

,text,label,start,end,start_document,end_document,score
0,HMS Prince of Wales,PRODUCT,38,57,9052,9071,None
1,HMS Repulse,PRODUCT,77,88,9091,9102,None
2,four,CARDINAL,93,97,9107,9111,None
3,Singapore,GPE,130,139,9144,9153,None
4,8 December,DATE,143,153,9157,9167,None
...,...,...,...,...,...,...,...
65,Johore,GPE,2463,2469,11477,11483,None
66,31 January,DATE,2475,2485,11489,11499,None
67,Commonwealth,ORG,2496,2508,11510,11522,None
68,Johor,GPE,2545,2550,11559,11564,None


הרצה על 100פסקאות

In [5]:
spacy_results = []

for number, paragraph in enumerate(sample, start=1):
    try:
        mentions = extract_spacy_mentions(paragraph)
        error = None
    except Exception as exc:
        mentions = []
        error = repr(exc)
        print(f"שגיאה בפסקה {number}: {error}")

    spacy_results.append({
        "paragraph": paragraph,
        "mentions": mentions,
        "error": error,
    })

    if number % 10 == 0:
        print(f"עובדו {number}/{len(sample)} פסקאות")

print("מספר פסקאות בתוצאות:", len(spacy_results))

עובדו 10/100 פסקאות
עובדו 20/100 פסקאות
עובדו 30/100 פסקאות
עובדו 40/100 פסקאות
עובדו 50/100 פסקאות
עובדו 60/100 פסקאות
עובדו 70/100 פסקאות
עובדו 80/100 פסקאות
עובדו 90/100 פסקאות
עובדו 100/100 פסקאות
מספר פסקאות בתוצאות: 100


שמירה התוצאות

In [6]:
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

output_dir = Path("/content/TMP_NER")
local_sample_path = output_dir / Path(sample_path).name

if not local_sample_path.is_file():
    raise FileNotFoundError(f"קובץ המדגם לא נמצא: {local_sample_path}")

if len(spacy_results) != sample_data["sample_size"]:
    raise ValueError("ההרצה לא הסתיימה על כל המדגם")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid.uuid4().hex[:8]

result_path = output_dir / (
    f"ner_en_core_web_trf_{local_sample_path.stem}_{run_id}.json"
)

output = {
    "model_id": SPACY_MODEL_ID,
    "revision_model": spacy_nlp.meta["version"],
    "spacy_version": spacy.__version__,
    "labels": sorted(spacy_nlp.get_pipe("ner").labels),
    "sample_file": local_sample_path.name,
    "seed": sample_data["seed"],
    "results": spacy_results,
}

with result_path.open("x", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2)

print("התוצאות נשמרו:", result_path)

התוצאות נשמרו: /content/TMP_NER/ner_en_core_web_trf_sample_100_paragraphs_seed42_20261010T095600Z_dfa6f0db.json


GLiNER

התקנת GLINER

In [7]:
%pip -q install "gliner2[local]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.8/355.8 kB 17.3 MB/s eta 0:00:00


תא 2 — טעינת המודל

In [8]:
from importlib.metadata import version
from gliner2 import AutoExtractor
from huggingface_hub import model_info

for package in ["gliner2", "transformers", "huggingface-hub", "torch"]:
    print(package, version(package))

print("GLiNER והחיבור ל-Hugging Face נטענו בהצלחה")

gliner2 2.1.0
transformers 5.18.0
huggingface-hub 1.33.0
torch 2.11.0+cpu
GLiNER והחיבור ל-Hugging Face נטענו בהצלחה


In [9]:
from huggingface_hub import model_info, snapshot_download
from gliner2 import AutoExtractor

GLINER_MODEL_ID = "fastino/gliner2.5-base-v1"
GLINER_REVISION = model_info(GLINER_MODEL_ID).sha

model_dir = snapshot_download(
    GLINER_MODEL_ID,
    revision=GLINER_REVISION,
)

gliner_model = AutoExtractor.from_pretrained(model_dir)

print("המודל נטען:", GLINER_MODEL_ID)
print("גרסה:", GLINER_REVISION)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[transformers] You are using a model of type `gliner2` to instantiate a model of type `extractor`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
[transformers] You are using a model of type `gliner2` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


המודל נטען: fastino/gliner2.5-base-v1
גרסה: 54785d51df8d86a0d8f2eb212fec15871aad2c4d


תא 3 — התוויות שנבקש בניסוי

In [10]:
GLINER_LABELS = [
    "person",
    "organization",
    "military unit",
    "location",
    "geopolitical entity",
    "facility",
    "nationality or political group",
    "event",
    "date",
    "time",
    "weapon",
    "vehicle",
    "product",
    "work of art",
    "law",
    "language",
    "money",
    "quantity",
    "percent",
    "ordinal",
    "cardinal",
    "miscellaneous named entity",
]

print("מספר תוויות מבוקשות:", len(GLINER_LABELS))

מספר תוויות מבוקשות: 22


תא 4 — חילוץ מפסקה אחת

In [11]:
def extract_gliner_mentions(paragraph):
    text = paragraph["text"]

    raw = gliner_model.extract_entities_long(
        text,
        GLINER_LABELS,
        chunk_size=384,
        chunk_overlap=64,
        threshold=0.5,
        include_spans=True,
        include_confidence=True,
        overlap_policy="allow",
    )

    mentions = []

    for label, entities in raw["entities"].items():
        for entity in entities:
            start = int(entity["start"])
            end = int(entity["end"])

            valid = (
                0 <= start < end <= len(text)
                and text[start:end] == entity["text"]
            )

            mentions.append({
                "text": text[start:end] if valid else entity["text"],
                "label": label,
                "start": start,
                "end": end,
                "start_document": paragraph["start"] + start if valid else None,
                "end_document": paragraph["start"] + end if valid else None,
                "score": entity.get("confidence"),
                "status_validation": "valid" if valid else "invalid",
            })

    return mentions, raw

תא 5 — בדיקה על פסקה אחת

In [12]:
import json
from pathlib import Path

sample_path = Path("/content/TMP_NER/sample_100_paragraphs_seed42.json")

with sample_path.open(encoding="utf-8") as file:
    sample_data = json.load(file)

sample = sample_data["paragraphs"]
print("נטענו", len(sample), "פסקאות")

נטענו 100 פסקאות


In [13]:
import pandas as pd

gliner_mentions_first, gliner_raw_first = extract_gliner_mentions(sample[0])

print("אזכורים שנמצאו:", len(gliner_mentions_first))
pd.DataFrame(gliner_mentions_first)

אזכורים שנמצאו: 51


,text,label,start,end,start_document,end_document,score,status_validation
0,Anderson,person,2038,2046,11052,11060,0.988673,valid
1,Lieutenant-Colonel John Parkin,person,2136,2166,11150,11180,0.624907,valid
2,John Parkin,person,2155,2166,11169,11180,0.883258,valid
3,Brigadier Harold Taylor,person,2396,2419,11410,11433,0.548039,valid
4,Harold Taylor,person,2406,2419,11420,11433,0.907252,valid
5,East Force,organization,2346,2356,11360,11370,0.915363,valid
6,8th Australian Division,military unit,954,977,9968,9991,0.959138,valid
7,2/30th Australian Battalion,military unit,1376,1403,10390,10417,0.970751,valid
8,45th Indian Brigade,military unit,1588,1607,10602,10621,0.973493,valid
9,5/11th Sikh Regiment,military unit,2114,2134,11128,11148,0.993465,valid


תא 6 — הרצה על אותן 100 פסקאות

In [14]:
gliner_results = []

for number, paragraph in enumerate(sample, start=1):
    try:
        mentions, raw = extract_gliner_mentions(paragraph)
        error = None
    except Exception as exc:
        mentions = []
        raw = None
        error = repr(exc)
        print(f"שגיאה בפסקה {number}: {error}")

    gliner_results.append({
        "paragraph": paragraph,
        "mentions": mentions,
        "pipeline_output": raw,
        "error": error,
    })

    if number % 10 == 0:
        print(f"עובדו {number}/{len(sample)} פסקאות")

print("מספר פסקאות בתוצאות:", len(gliner_results))
print("פסקאות עם שגיאה:", sum(r["error"] is not None for r in gliner_results))

עובדו 10/100 פסקאות
עובדו 20/100 פסקאות
עובדו 30/100 פסקאות
עובדו 40/100 פסקאות
עובדו 50/100 פסקאות
עובדו 60/100 פסקאות
עובדו 70/100 פסקאות
עובדו 80/100 פסקאות
עובדו 90/100 פסקאות
עובדו 100/100 פסקאות
מספר פסקאות בתוצאות: 100
פסקאות עם שגיאה: 0


תא 7 — שמירה ב־/content/TMP_NER

In [15]:
import hashlib
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

output_dir = Path("/content/TMP_NER")
local_sample_path = output_dir / Path(sample_path).name

if not local_sample_path.is_file():
    raise FileNotFoundError(local_sample_path)

if len(gliner_results) != sample_data["sample_size"]:
    raise ValueError("ההרצה לא הסתיימה על כל המדגם")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid.uuid4().hex[:8]

result_path = output_dir / (
    f"ner_fastino_gliner2_5_base_v1_"
    f"{local_sample_path.stem}_{run_id}.json"
)

output = {
    "schema_version": "0.1.0-experiment",
    "model_id": GLINER_MODEL_ID,
    "revision_model": GLINER_REVISION,
    "sample_file": local_sample_path.name,
    "sample_sha256": hashlib.sha256(local_sample_path.read_bytes()).hexdigest(),
    "seed": sample_data["seed"],
    "requested_labels": GLINER_LABELS,
    "threshold": 0.5,
    "chunk_size_words": 384,
    "chunk_overlap_words": 64,
    "overlap_policy": "allow",
    "results": gliner_results,
}

def json_number(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot save {type(value).__name__}")

with result_path.open("x", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2,
              default=json_number)

print("התוצאות נשמרו:", result_path)

התוצאות נשמרו: /content/TMP_NER/ner_fastino_gliner2_5_base_v1_sample_100_paragraphs_seed42_20261010T100041Z_ee8569b7.json


FLAIR

התקנת FLAIR

In [16]:
%pip install -q flair

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.1/203.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

תא 3 — טעינת המודל

In [1]:
from flair.models import SequenceTagger
from flair.splitter import SegtokSentenceSplitter
from huggingface_hub import hf_hub_download, model_info

FLAIR_MODEL_ID = "flair/ner-english-ontonotes-large"
FLAIR_REVISION = model_info(FLAIR_MODEL_ID).sha

model_file = hf_hub_download(
    repo_id=FLAIR_MODEL_ID,
    filename="pytorch_model.bin",
    revision=FLAIR_REVISION,
)

flair_ner = SequenceTagger.load(model_file)
flair_splitter = SegtokSentenceSplitter()

print("המודל נטען:", FLAIR_MODEL_ID)
print("גרסת המודל:", FLAIR_REVISION)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2026-10-10 10:05:39,175 SequenceTagger predicts: Dictionary with 76 tags: <unk>, O, B-CARDINAL, E-CARDINAL, S-PERSON, S-CARDINAL, S-PRODUCT, B-PRODUCT, I-PRODUCT, E-PRODUCT, B-WORK_OF_ART, I-WORK_OF_ART, E-WORK_OF_ART, B-PERSON, E-PERSON, S-GPE, B-DATE, I-DATE, E-DATE, S-ORDINAL, S-LANGUAGE, I-PERSON, S-EVENT, S-DATE, B-QUANTITY, E-QUANTITY, S-TIME, B-TIME, I-TIME, E-TIME, B-GPE, E-GPE, S-ORG, I-GPE, S-NORP, B-FAC, I-FAC, E-FAC, B-NORP, E-NORP, S-PERCENT, B-ORG, E-ORG, B-LANGUAGE, E-LANGUAGE, I-CARDINAL, I-ORG, S-WORK_OF_ART, I-QUANTITY, B-MONEY
המודל נטען: flair/ner-english-ontonotes-large
גרסת המודל: 4ffb3596f4359f0c8799ea15bbf5dbb3b0915a53


תא 2 — טעינת אותו מדגם

In [4]:
import json
from pathlib import Path

sample_path = Path("/content/TMP_NER/sample_100_paragraphs_seed42.json")

with sample_path.open(encoding="utf-8") as file:
    sample_data = json.load(file)

sample = sample_data["paragraphs"]
print(f"נטענו {len(sample)} פסקאות מהקובץ {sample_path.name}")

נטענו 100 פסקאות מהקובץ sample_100_paragraphs_seed42.json


תא 4 — פונקציה לפסקה אחת

In [2]:
def _collapse_whitespace_with_map(text):
    normalized = []
    source_starts = []
    source_ends = []

    position = 0
    while position < len(text):
        if text[position].isspace():
            end = position + 1
            while end < len(text) and text[end].isspace():
                end += 1

            normalized.append(" ")
            source_starts.append(position)
            source_ends.append(end)
            position = end
        else:
            normalized.append(text[position])
            source_starts.append(position)
            source_ends.append(position + 1)
            position += 1

    return "".join(normalized), source_starts, source_ends


def _collapse_whitespace(text):
    return _collapse_whitespace_with_map(text)[0]


def _invalid_flair_mention(entity, error):
    return {
        "text": None,
        "flair_text": entity["text"],
        "label": entity["label"],
        "start": None,
        "end": None,
        "start_document": None,
        "end_document": None,
        "score": entity["score"],
        "status_validation": "invalid",
        "error": error,
    }


def extract_flair_mentions(paragraph):
    paragraph_text = paragraph["text"]
    sentences = flair_splitter.split(paragraph_text)

    if sentences:
        flair_ner.predict(sentences, mini_batch_size=1)

    # קודם שומרים את פלט Flair. ההתאמה להיסטים נעשית רק אחר כך.
    raw_output = []
    for sentence in sentences:
        entities = []

        for span in sentence.get_spans("ner"):
            label = span.get_label("ner")
            entities.append({
                "text": span.text,
                "start_pos": span.start_position,
                "end_pos": span.end_position,
                "label": label.value,
                "score": float(label.score),
            })

        raw_output.append({
            "sentence_text": sentence.text,
            "sentence_start_flair": sentence.start_position,
            "entities": entities,
            "flair_output": sentence.to_dict(),
        })

    normalized_source, source_starts, source_ends = (
        _collapse_whitespace_with_map(paragraph_text)
    )

    mentions = []
    cursor = 0

    for sentence_number, record in enumerate(raw_output):
        sentence_text = record["sentence_text"]
        normalized_sentence = _collapse_whitespace(sentence_text)

        sentence_start = normalized_source.find(normalized_sentence, cursor)

        if not normalized_sentence or sentence_start < 0:
            for entity in record["entities"]:
                mentions.append(_invalid_flair_mention(
                    entity,
                    f"משפט {sentence_number}: אי אפשר להתאים את המשפט לטקסט המקורי"
                ))
            continue

        cursor = sentence_start + len(normalized_sentence)

        for entity_number, entity in enumerate(record["entities"]):
            local_start = entity["start_pos"]
            local_end = entity["end_pos"]

            if not 0 <= local_start < local_end <= len(sentence_text):
                mentions.append(_invalid_flair_mention(
                    entity,
                    f"משפט {sentence_number}, אזכור {entity_number}: "
                    "היסטים מקומיים לא תקינים"
                ))
                continue

            normalized_start = (
                sentence_start
                + len(_collapse_whitespace(sentence_text[:local_start]))
            )
            normalized_end = (
                sentence_start
                + len(_collapse_whitespace(sentence_text[:local_end]))
            )

            if not 0 <= normalized_start < normalized_end <= len(normalized_source):
                mentions.append(_invalid_flair_mention(
                    entity,
                    f"משפט {sentence_number}, אזכור {entity_number}: "
                    "האזכור נמצא מחוץ לטקסט המקורי"
                ))
                continue

            start = source_starts[normalized_start]
            end = source_ends[normalized_end - 1]
            original_text = paragraph_text[start:end]

            if (
                _collapse_whitespace(original_text)
                != _collapse_whitespace(entity["text"])
            ):
                mentions.append(_invalid_flair_mention(
                    entity,
                    f"משפט {sentence_number}, אזכור {entity_number}: "
                    "טקסט האזכור אינו תואם למקור"
                ))
                continue

            mention = {
                "text": original_text,
                "label": entity["label"],
                "start": start,
                "end": end,
                "start_document": paragraph["start"] + start,
                "end_document": paragraph["start"] + end,
                "score": entity["score"],
                "status_validation": "pending",
            }

            assert paragraph_text[start:end] == mention["text"]
            assert mention["start_document"] == paragraph["start"] + start
            assert mention["end_document"] == paragraph["start"] + end

            mentions.append(mention)

    return mentions, raw_output

In [13]:
# def extract_flair_mentions(paragraph):
#     text = paragraph["text"]
#     sentences = flair_splitter.split(text)

#     if sentences:
#         flair_ner.predict(sentences, mini_batch_size=1)

#     mentions = []
#     raw_output = []

#     for sentence in sentences:
#         raw_output.append({
#             "sentence_start": sentence.start_position,
#             "flair_output": sentence.to_dict(),
#         })

#         for entity in sentence.get_spans("ner"):
#             start = sentence.start_position + entity.start_position
#             end = sentence.start_position + entity.end_position
#             label = entity.get_label("ner")

#             if text[start:end] != entity.text:
#                 raise ValueError(
#                     f"אי־התאמה במיקום האזכור: {entity.text!r} מול {text[start:end]!r}"
#                 )

#             mentions.append({
#                 "text": text[start:end],
#                 "label": label.value,
#                 "start": start,
#                 "end": end,
#                 "start_document": paragraph["start"] + start,
#                 "end_document": paragraph["start"] + end,
#                 "score": float(label.score),
#                 "status_validation": "pending",
#             })

#     return mentions, raw_output

תא 5 — בדיקה על פסקה אחת בלבד

In [5]:
import pandas as pd

flair_mentions_first, flair_raw_first = extract_flair_mentions(sample[0])

print("מספר אזכורים בפסקה הראשונה:", len(flair_mentions_first))
display(pd.DataFrame(flair_mentions_first))

מספר אזכורים בפסקה הראשונה: 72


,text,label,start,end,start_document,end_document,score,status_validation
0,Force Z,ORG,0,7,9014,9021,0.998097,pending
1,HMS Prince of Wales,PRODUCT,38,57,9052,9071,0.999960,pending
2,HMS Repulse,PRODUCT,77,88,9091,9102,0.999944,pending
3,four,CARDINAL,93,97,9107,9111,0.999984,pending
4,Singapore,GPE,130,139,9144,9153,0.999995,pending
...,...,...,...,...,...,...,...,...
67,Johore,GPE,2463,2469,11477,11483,0.999993,pending
68,31 January,DATE,2475,2485,11489,11499,0.999985,pending
69,Commonwealth,ORG,2496,2508,11510,11522,0.999351,pending
70,Johor,GPE,2545,2550,11559,11564,0.999997,pending


תא 6 — הרצה על כל 100 הפסקאות

In [7]:
flair_results_v2 = []

for index, paragraph in enumerate(sample):
    try:
        mentions, raw_output = extract_flair_mentions(paragraph)

        errors = [
            mention["error"] for mention in mentions
            if mention["status_validation"] == "invalid"
        ]

        status = "success" if not errors else "alignment_error"
        error = "; ".join(errors) if errors else None

    except Exception as exc:
        mentions = []
        raw_output = []
        status = "error"
        error = f"{type(exc).__name__}: {exc}"

    flair_results_v2.append({
        "sample_index": index,
        "article_title": paragraph["article_title"],
        "paragraph_index": paragraph["paragraph_index"],
        "start": paragraph["start"],
        "end": paragraph["end"],
        "text": paragraph["text"],
        "mentions": mentions,
        "raw_output": raw_output,
        "status_processing": status,
        "error": error,
    })

print("פסקאות שעובדו:", len(flair_results_v2))
print("הצלחות:", sum(r["status_processing"] == "success" for r in flair_results_v2))
print("שגיאות התאמה:", sum(r["status_processing"] == "alignment_error" for r in flair_results_v2))
print("שגיאות הרצה:", sum(r["status_processing"] == "error" for r in flair_results_v2))


# flair_results = []

# for index, paragraph in enumerate(sample):
#     try:
#         mentions, raw_output = extract_flair_mentions(paragraph)
#         status = "success"
#         error = None
#     except Exception as exc:
#         mentions = []
#         raw_output = []
#         status = "error"
#         error = f"{type(exc).__name__}: {exc}"

#     flair_results.append({
#         "sample_index": index,
#         "article_title": paragraph["article_title"],
#         "paragraph_index": paragraph["paragraph_index"],
#         "start": paragraph["start"],
#         "end": paragraph["end"],
#         "text": paragraph["text"],
#         "mentions": mentions,
#         "raw_output": raw_output,
#         "status_processing": status,
#         "error": error,
#     })

# print("פסקאות שעובדו:", len(flair_results))
# print("פסקאות עם שגיאה:", sum(r["status_processing"] == "error" for r in flair_results))
# print("סך האזכורים:", sum(len(r["mentions"]) for r in flair_results))

פסקאות שעובדו: 100
הצלחות: 100
שגיאות התאמה: 0
שגיאות הרצה: 0


תא 7 — שמירה לצד המדגם, בשם ייחודי ל־Flair

In [9]:
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version
from uuid import uuid4

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
flair_output_path_v2 = sample_path.parent / f"flair_ontonotes_large_aligned_{run_id}.json"

result_data_v2 = {
    "schema_version": "experiment-2",
    "name_model": FLAIR_MODEL_ID,
    "revision_model": FLAIR_REVISION,
    "flair_version": version("flair"),
    "sample_file": sample_path.name,
    "sample_sha256": hashlib.sha256(sample_path.read_bytes()).hexdigest(),
    "source_file": sample_data.get("source_file"),
    "seed": sample_data.get("seed"),
    "sample_size": len(sample),
    "sentence_splitter": "SegtokSentenceSplitter",
    "alignment_method": "sequential_whitespace_equivalence",
    "at_processed": datetime.now(timezone.utc).isoformat(),
    "results": flair_results_v2,
}

with flair_output_path_v2.open("x", encoding="utf-8") as file:
    json.dump(result_data_v2, file, ensure_ascii=False, indent=2)

print("נשמר קובץ חדש:", flair_output_path_v2)
# import hashlib
# from datetime import datetime, timezone
# from importlib.metadata import version
# from uuid import uuid4

# output_dir = sample_path.parent
# run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
# flair_output_path = output_dir / f"flair_ontonotes_large_{run_id}.json"

# result_data = {
#     "schema_version": "experiment-1",
#     "name_model": FLAIR_MODEL_ID,
#     "revision_model": FLAIR_REVISION,
#     "flair_version": version("flair"),
#     "sample_file": sample_path.name,
#     "sample_sha256": hashlib.sha256(sample_path.read_bytes()).hexdigest(),
#     "seed": sample_data.get("seed"),
#     "sample_size": len(sample),
#     "sentence_splitter": "SegtokSentenceSplitter",
#     "at_processed": datetime.now(timezone.utc).isoformat(),
#     "results": flair_results,
# }

# with flair_output_path.open("w", encoding="utf-8") as file:
#     json.dump(result_data, file, ensure_ascii=False, indent=2)

# print("התוצאות נשמרו ב־", flair_output_path)

נשמר קובץ חדש: /content/TMP_NER/flair_ontonotes_large_aligned_20261010T110233Z_847d815b.json


תא אימות

In [10]:
with flair_output_path_v2.open(encoding="utf-8") as file:
    saved = json.load(file)

assert len(saved["results"]) == len(sample) == 100
assert saved["sample_sha256"] == hashlib.sha256(sample_path.read_bytes()).hexdigest()

mention_count = 0

for index, (paragraph, result) in enumerate(zip(sample, saved["results"])):
    assert result["sample_index"] == index
    assert result["status_processing"] == "success"

    for field in ["article_title", "paragraph_index", "start", "end", "text"]:
        assert result[field] == paragraph[field], (index, field)

    for mention in result["mentions"]:
        assert mention["status_validation"] != "invalid", index

        start = mention["start"]
        end = mention["end"]

        assert 0 <= start < end <= len(paragraph["text"]), index
        assert paragraph["text"][start:end] == mention["text"], index
        assert mention["start_document"] == paragraph["start"] + start, index
        assert mention["end_document"] == paragraph["start"] + end, index

        mention_count += 1

print("האימות עבר: 100 פסקאות, ללא שגיאות והיסטים תקינים.")
print("מספר אזכורים:", mention_count)
print("הקובץ:", flair_output_path_v2)

האימות עבר: 100 פסקאות, ללא שגיאות והיסטים תקינים.
מספר אזכורים: 3127
הקובץ: /content/TMP_NER/flair_ontonotes_large_aligned_20261010T110233Z_847d815b.json


שמירת המדגם והתוצאות למחשב

In [11]:
from pathlib import Path
from shutil import make_archive
from datetime import datetime, timezone
from uuid import uuid4
from google.colab import files

output_dir = Path("/content/TMP_NER")

if not output_dir.is_dir() or not any(output_dir.iterdir()):
    raise FileNotFoundError("התיקייה /content/TMP_NER חסרה או ריקה")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid4().hex[:8]

archive_name = Path("/content") / f"NER_results_{run_id}"
zip_path = make_archive(
    str(archive_name),
    "zip",
    root_dir=str(output_dir.parent),
    base_dir=output_dir.name,
)

print("מוריד:", zip_path)
files.download(zip_path)


מוריד: /content/NER_results_20261010T110259Z_97a75c17.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>